In [1]:
# 从theo.ipynb 删减得到，精简完整实验版本
# 通过utils-exp_func重构

import numpy as np
import math
import matplotlib.pyplot as plt

import mujoco
import mujoco.viewer
import mediapy as media
import datetime
import pandas as pd
from tqdm import tqdm
import matplotlib.gridspec as gridspec
from matplotlib.widgets import Slider
import scipy
import ipywidgets as widgets
from IPython.display import display
from scipy.optimize import differential_evolution
from pathlib import Path
import pickle

import rootpath
import sys
root_dir = rootpath.detect()   # 自动查找根目录
sys.path.append(str(Path(root_dir)))
from utils.experiment import MujocoExperiment


In [2]:
# theta->static force (theoretical)
def get_static_func(theta_1, theta_2, k_3=637.52/2, k_4=631.6/2, l_10=0.174, l_20=0.252, m_3 = 0.18648, m_4=0.27266):
    """
        计算静力学
        - theta_1: 关节1角度
        - theta_2: 关节2角度
        - k_3: spring 1 stiffness
        - k_4: spring 2 stiffness
        - l_10: spring 1 original length
        - l_20: spring 2 original length
    """

    a_1 = 0.25
    a_2 = 0.25
    b_1 = 0.21213
    b_2 = 0.1
    d_1 = 0.06
    d_2 = 0.10
    # s = 0

    # M = 0.155  # 真实值
    m_1, m_2 = 0.086, 0.1033
    # I_1, I_2 = (1/12 * m_1 * 0.25**2), (1/12 * m_2 * 0.25**2)

    # m_3, m_4 = 0.18648, 0.27266

    # k_3, k_4 = 637.52/2, 631.6/2

    # 阻尼参数
    # c_3, c_4 = 22.68/2, 21.8/2

    # 重力加速度
    g = 9.8

    # 角度参数 (度转弧度)
    beta_1 = 8.13 / 180 * np.pi
    beta_2 = 30 / 180 * np.pi

    # 长度参数
    # l_10 = 0.174
    # l_20 = 0.252

    A_x_O = d_1
    A_y_O = 0

    B_x_O = -d_2
    B_y_O = 0

    C_x_O = b_1 * np.cos(theta_1 - beta_1)
    C_y_O = b_1 * np.sin(theta_1 - beta_1)

    D_x_O = a_1 * np.cos(theta_1) + b_2 * np.cos(theta_1 + theta_2 + beta_2)
    D_y_O = a_1 * np.sin(theta_1) + b_2 * np.sin(theta_1 + theta_2 + beta_2)

    E_x_O = a_1 * np.cos(theta_1)
    E_y_O = a_1 * np.sin(theta_1)

    F_x_O = a_1 * np.cos(theta_1) + a_2 * np.cos(theta_1 + theta_2)
    F_y_O = a_1 * np.sin(theta_1) + a_2 * np.sin(theta_1 + theta_2)

    G_x_O = b_1 * np.cos(beta_1) * np.cos(theta_1)
    G_y_O = b_1 * np.cos(beta_1) * np.sin(theta_1)

    H_x_O = a_1 * np.cos(theta_1) + b_2 * np.cos(beta_2) * np.cos(theta_1 + theta_2)
    H_y_O = a_1 * np.sin(theta_1) + b_2 * np.cos(beta_2) * np.sin(theta_1 + theta_2)

    # 计算偏导数
    # 对 theta_1 的偏导数

    dA_x_O_dtheta_1 = 0
    dA_y_O_dtheta_1 = 0

    dB_x_O_dtheta_1 = 0
    dB_y_O_dtheta_1 = 0

    dC_x_O_dtheta_1 = -b_1 * np.sin(theta_1 - beta_1)
    dC_y_O_dtheta_1 = b_1 * np.cos(theta_1 - beta_1)

    dD_x_O_dtheta_1 = -a_1 * np.sin(theta_1) - b_2 * np.sin(theta_1 + theta_2 + beta_2)
    dD_y_O_dtheta_1 = a_1 * np.cos(theta_1) + b_2 * np.cos(theta_1 + theta_2 + beta_2)

    dE_x_O_dtheta_1 = -a_1 * np.sin(theta_1)
    dE_y_O_dtheta_1 = a_1 * np.cos(theta_1)

    dF_x_O_dtheta_1 = -a_1 * np.sin(theta_1) - a_2 * np.sin(theta_1 + theta_2)
    dF_y_O_dtheta_1 = a_1 * np.cos(theta_1) + a_2 * np.cos(theta_1 + theta_2)

    dG_x_O_dtheta_1 = -b_1 * np.cos(beta_1) * np.sin(theta_1)
    dG_y_O_dtheta_1 = b_1 * np.cos(beta_1) * np.cos(theta_1)

    dH_x_O_dtheta_1 = -a_1 * np.sin(theta_1) - b_2 * np.cos(beta_2) * np.sin(theta_1 + theta_2)
    dH_y_O_dtheta_1 = a_1 * np.cos(theta_1) + b_2 * np.cos(beta_2) * np.cos(theta_1 + theta_2)

    # 对 theta_2 的偏导数

    dA_x_O_dtheta_2 = 0
    dA_y_O_dtheta_2 = 0

    dB_x_O_dtheta_2 = 0
    dB_y_O_dtheta_2 = 0

    dC_x_O_dtheta_2 = 0
    dC_y_O_dtheta_2 = 0

    dD_x_O_dtheta_2 = -b_2 * np.sin(theta_1 + theta_2 + beta_2)
    dD_y_O_dtheta_2 = b_2 * np.cos(theta_1 + theta_2 + beta_2)

    dE_x_O_dtheta_2 = 0
    dE_y_O_dtheta_2 = 0

    dF_x_O_dtheta_2 = -a_2 * np.sin(theta_1 + theta_2)
    dF_y_O_dtheta_2 = a_2 * np.cos(theta_1 + theta_2)

    dH_x_O_dtheta_2 = -b_2 * np.cos(beta_2) * np.sin(theta_1 + theta_2)
    dH_y_O_dtheta_2 = b_2 * np.cos(beta_2) * np.cos(theta_1 + theta_2)


    # 计算长度 l1 和 l2
    l_1 = np.sqrt((A_x_O - C_x_O)**2 + (A_y_O - C_y_O)**2)
    l_2 = np.sqrt((B_x_O - D_x_O)**2 + (B_y_O - D_y_O)**2)

    # 计算偏导数
    # 偏导数 d/dtheta_1
    dl_1_dtheta_1 = 1/l_1 * ((A_x_O - C_x_O) * (dA_x_O_dtheta_1 - dC_x_O_dtheta_1) + (A_y_O - C_y_O) * (dA_y_O_dtheta_1 - dC_y_O_dtheta_1))
    dl_2_dtheta_1 = 1/l_2 * ((B_x_O - D_x_O) * (dB_x_O_dtheta_1 - dD_x_O_dtheta_1) + (B_y_O - D_y_O) * (dB_y_O_dtheta_1 - dD_y_O_dtheta_1))

    # 偏导数 d/dtheta_2
    dl_1_dtheta_2 = 1/l_1 * ((A_x_O - C_x_O) * (dA_x_O_dtheta_2 - dC_x_O_dtheta_2) + (A_y_O - C_y_O) * (dA_y_O_dtheta_2 - dC_y_O_dtheta_2))
    dl_2_dtheta_2 = 1/l_2 * ((B_x_O - D_x_O) * (dB_x_O_dtheta_2 - dD_x_O_dtheta_2) + (B_y_O - D_y_O) * (dB_y_O_dtheta_2 - dD_y_O_dtheta_2))

    # print(dl_1_dtheta_2)  # check the model

    # 等式右侧
    RHSb_1 = -( m_1*g*(dE_y_O_dtheta_1/2) + m_2*g*((dE_y_O_dtheta_1+dF_y_O_dtheta_1)/2) + m_3*g*(dC_y_O_dtheta_1/2) + m_4*g*(dD_y_O_dtheta_1/2) )
    RHSb_2 = -( m_1*g*(dE_y_O_dtheta_2/2) + m_2*g*((dE_y_O_dtheta_2+dF_y_O_dtheta_2)/2) + m_3*g*(dC_y_O_dtheta_2/2) + m_4*g*(dD_y_O_dtheta_2/2) )
    b = np.array([RHSb_1, RHSb_2])

    # 等式左侧
    LHSA = np.array([[dl_1_dtheta_1, dl_2_dtheta_1], 
                    [dl_1_dtheta_2, dl_2_dtheta_2]])

    # 回复力
    F_k = np.array([k_3*(l_1-l_10), k_4*(l_2-l_20)])

    # 计算静力学
    StaticForce = np.linalg.solve(LHSA, b) + F_k
    return StaticForce

get_static_func(math.pi/6, math.pi/2*0, 637.52/2, 631.6/2, 0.174, 0.252)

array([-31.07359493,  57.5943789 ])

In [3]:
# Copy from <dynamic_identify.ipynb>
# Tools: 计算长度偏置（初始长度-原长）
def l_calulator(theta_1, theta_2):
    a_1 = 0.25
    a_2 = 0.25
    b_1 = 0.21213
    b_2 = 0.1
    d_1 = 0.06
    d_2 = 0.10

    # 角度参数 (度转弧度)
    beta_1 = 8.13 / 180 * np.pi
    beta_2 = 30 / 180 * np.pi

    A_x_O = d_1
    A_y_O = 0

    B_x_O = -d_2
    B_y_O = 0

    C_x_O = b_1 * np.cos(theta_1 - beta_1)
    C_y_O = b_1 * np.sin(theta_1 - beta_1)

    D_x_O = a_1 * np.cos(theta_1) + b_2 * np.cos(theta_1 + theta_2 + beta_2)
    D_y_O = a_1 * np.sin(theta_1) + b_2 * np.sin(theta_1 + theta_2 + beta_2)

    # 计算长度 l1 和 l2
    l_1 = np.sqrt((A_x_O - C_x_O)**2 + (A_y_O - C_y_O)**2)
    l_2 = np.sqrt((B_x_O - D_x_O)**2 + (B_y_O - D_y_O)**2)

    return l_1, l_2

# Tools: 计算驱动器设置初始偏置，这里采用真实模型
def bias_calculator(l_10, l_20):
  l_1, l_2 = l_calulator(np.pi/2, np.pi/2)      # 真实模型
  l1_rel = l_10 - l_1
  l2_rel = l_20 - l_2
  return l1_rel/2, l2_rel/2

# 从/utils/exp_func/experiment.py中调用函数并进行对比
theta1_test = np.random.uniform(0, np.pi/2, 100)
theta2_test = np.random.uniform(0, np.pi/2, 100)

for i in range(len(theta1_test)):
    l1, l2 = l_calulator(theta1_test[i], theta2_test[i])
    l1_, l2_ = MujocoExperiment.calculate_length(theta1_test[i], theta2_test[i])
    assert math.isclose(l1, l1_, rel_tol=1e-9), f"l1: {l1} != {l1_}"
    assert math.isclose(l2, l2_, rel_tol=2e-9), f"l2: {l2} != {l2_}"

## Video Recording - A case of simulated verification

In [5]:
# A demo for video making
np.random.seed(42)
path = (root_dir + "/models/v2_4/urdf/dog2_4singleLeg_realconstrast.xml")
exp = MujocoExperiment(path)
para_init = {
    'stiffness_MAA': 318.76, #637.52 / 2,
    'stiffness_BAA': 315.8, #631.6 / 2,
    'l10': 0.174,
    'l20': 0.252,
    'damping_MAA': 11.34,
    'damping_BAA': 10.90,
    'c1_thigh': 0,
    'c2_calf': 0,
    's1': 1.0,
    's2': 1.0,
    'P1': 0,        # To be set
    'P2': 0,
    'P1_prime': 0.0,      
    'P2_prime': 0.0       # equal to F when s=1
}

para_record_list = []
exp_data = []
frames_last_group = None

theta_1_tar_array = np.linspace(math.pi/6, math.pi*2/3, 11)[1:]
theta_2_tar_array = np.linspace(0, math.pi/2, 11)[1:]

# set parameter
para = para_init.copy()     # create new parameter
para['stiffness_MAA'] *= np.random.uniform(0.5, 1.5)
para['stiffness_BAA'] *= np.random.uniform(0.5, 1.5)
para['l10'] *= np.random.uniform(0.5, 1.5)
para['l20'] *= np.random.uniform(0.5, 1.5)
para_record_list.append(para)
print(f"Para: {para}")


theta_1_tar_array = np.linspace(math.pi/6, math.pi*2/3, 11)[1:]
theta_2_tar_array = np.linspace(0, math.pi/2, 11)[1:]
for i in tqdm(range(len(theta_1_tar_array))):
    for j in range(len(theta_2_tar_array)):
        theta_1_theo = theta_1_tar_array[i]
        theta_2_theo = theta_2_tar_array[j]
        force_theoretical = get_static_func(theta_1_theo, theta_2_theo, para['stiffness_MAA'], para['stiffness_BAA'], para['l10'], para['l20'])
        para['P1_prime'] = force_theoretical[0]
        para['P2_prime'] = force_theoretical[1]
        time_sim_, theta1_sim_, theta2_sim_, frames_, valid_, valid_last_ = exp.run(para, time_step=0, duration=10, ifrender=True)
        # Save Video
        media.write_video(f"./video/experiment_output_{i*10+j}.mp4", frames_, fps=60)

        exp_data.append({
            'valid': valid_,
            'valid_last': valid_last_,
            'theta_1_theo': theta_1_theo,
            'theta_2_theo': theta_2_theo,
            'theta1_sim_': theta1_sim_[-1],
            'theta2_sim_': theta2_sim_[-1],
            'F1_apply': force_theoretical[0],
            'F2_apply': force_theoretical[1],
        })
            


Para: {'stiffness_MAA': 278.7684082837853, 'stiffness_BAA': 458.1355779642515, 'l10': 0.21436694587518446, 'l20': 0.27686193801765324, 'damping_MAA': 11.34, 'damping_BAA': 10.9, 'c1_thigh': 0, 'c2_calf': 0, 's1': 1.0, 's2': 1.0, 'P1': 0, 'P2': 0, 'P1_prime': 0.0, 'P2_prime': 0.0}


100%|██████████| 10/10 [08:26<00:00, 50.65s/it]


In [6]:
exp_data = pd.DataFrame(data=exp_data)
exp_data.to_csv("./data/demo/experiment_data.csv", index=False)
para_record_list = pd.DataFrame(data=para_record_list)
para_record_list.to_csv("./data/demo/experiment_para.csv", index=False)

## Static Response of Multi Group of parameteres

In [7]:
def run_one_experiment(exp, para):
    '''one group of parameter, scan 10*10 Force Commands'''
    exp_data = []

    for i in tqdm(range(len(theta_1_tar_array))):
        for j in range(len(theta_2_tar_array)):
            theta_1_theo = theta_1_tar_array[i]
            theta_2_theo = theta_2_tar_array[j]
            force_theoretical = get_static_func(theta_1_theo, theta_2_theo, para['stiffness_MAA'], para['stiffness_BAA'], para['l10'], para['l20'])
            para['P1_prime'] = force_theoretical[0]
            para['P2_prime'] = force_theoretical[1]
            time_sim_, theta1_sim_, theta2_sim_, frames_, valid_, valid_last_ = exp.run(para, time_step=0, duration=10, ifrender=False)
            # Save Video
            # media.write_video(f'./video/experiment_output_{i*10+j}.mp4', frames_, fps=60)

            if valid_last_:
                # record the data
                exp_data.append({
                    'theta_1_theo': theta_1_theo,
                    'theta_2_theo': theta_2_theo,
                    'theta_1_sim': theta1_sim_[-1],
                    'theta_2_sim': theta2_sim_[-1],
                    'F1_apply': force_theoretical[0],
                    'F2_apply': force_theoretical[1],
                })
    exp_data = pd.DataFrame(data=exp_data)
    # cal error from valid data
    theta_1_theo_array = exp_data['theta_1_theo'].to_numpy()
    theta_2_theo_array = exp_data['theta_2_theo'].to_numpy()
    theta_1_sim_array = exp_data['theta_1_sim'].to_numpy()
    theta_2_sim_array = exp_data['theta_2_sim'].to_numpy()
    # 计算 RMSE, MAE, MaxAE
    RMSE_theta_1 = np.sqrt(np.mean((theta_1_theo_array - theta_1_sim_array)**2))
    RMSE_theta_2 = np.sqrt(np.mean((theta_2_theo_array - theta_2_sim_array)**2))
    MAE_theta_1 = np.mean(np.abs(theta_1_theo_array - theta_1_sim_array))
    MAE_theta_2 = np.mean(np.abs(theta_2_theo_array - theta_2_sim_array))
    MaxAE_theta_1 = np.max(np.abs(theta_1_theo_array - theta_1_sim_array))
    MaxAE_theta_2 = np.max(np.abs(theta_2_theo_array - theta_2_sim_array))
    return {
        'k3': para['stiffness_MAA'],
        'k4': para['stiffness_BAA'],
        'l10': para['l10'],
        'l20': para['l20'],
        'RMSE_theta_1': RMSE_theta_1,
        'RMSE_theta_2': RMSE_theta_2,
        'MAE_theta_1': MAE_theta_1,
        'MAE_theta_2': MAE_theta_2,
        'MaxAE_theta_1': MaxAE_theta_1,
        'MaxAE_theta_2': MaxAE_theta_2,
        'valid_exp_num': len(exp_data)
    }

def run_batch_experiments():
    exp = MujocoExperiment(path)
    para_init = {
        'stiffness_MAA': 318.76, #637.52 / 2,
        'stiffness_BAA': 315.8, #631.6 / 2,
        'l10': 0.174,
        'l20': 0.252,
        'damping_MAA': 11.34,
        'damping_BAA': 10.90,
        'c1_thigh': 0,
        'c2_calf': 0,
        's1': 1.0,
        's2': 1.0,
        'P1': 0,        # To be set
        'P2': 0,
        'P1_prime': 0.0,      
        'P2_prime': 0.0       # equal to F when s=1
    }
    multi_exp_data = []

    for i in range(100):
        # randomly set stiffness and l10, l20
        para = para_init.copy()     # create new parameter
        para['stiffness_MAA'] *= np.random.uniform(0.5, 1.5)
        para['stiffness_BAA'] *= np.random.uniform(0.5, 1.5)
        para['l10'] *= np.random.uniform(0.5, 1.5)
        para['l20'] *= np.random.uniform(0.5, 1.5)

        # run one experiment
        single_experiment_res_dic = run_one_experiment(exp, para)
        if single_experiment_res_dic['valid_exp_num'] == 0:
            print("No valid experiment data.")
            continue
        multi_exp_data.append(single_experiment_res_dic)
    return multi_exp_data

In [8]:
np.random.seed(42)
multi_exp_data = run_batch_experiments()
multi_exp_data = pd.DataFrame(data=multi_exp_data)
multi_exp_data.to_csv("./data/multi_exps_data.csv", index=False)

100%|██████████| 10/10 [00:11<00:00,  1.17s/it]


## Plot the result

In [9]:
# Load the data
multi_exp_data = pd.read_csv("./data/multi_exps_data.csv")
multi_exp_data

,k3,k4,l10,l20,RMSE_theta_1,RMSE_theta_2,MAE_theta_1,MAE_theta_2,MaxAE_theta_1,MaxAE_theta_2,valid_exp_num
0,278.768408,458.135578,0.214367,0.276862,0.001146,0.003825,0.000949,0.003366,0.003314,0.010166,100
1,209.112502,207.163070,0.097107,0.344276,0.001405,0.007348,0.001122,0.006585,0.004990,0.019874,100
2,350.991421,381.509320,0.090582,0.370417,0.001100,0.004278,0.000891,0.003858,0.003096,0.011025,100
3,424.729416,224.956691,0.118638,0.172218,0.000977,0.006475,0.000789,0.005836,0.003377,0.016331,100
4,256.360257,323.618081,0.162158,0.199390,0.001170,0.005022,0.000930,0.004464,0.003864,0.013384,100
...,...,...,...,...,...,...,...,...,...,...,...
95,197.046220,377.929597,0.196436,0.347123,0.001377,0.004628,0.001126,0.004143,0.004078,0.012501,100
96,393.691246,411.639278,0.136074,0.170715,0.000942,0.003950,0.000785,0.003548,0.002883,0.010198,100
97,398.645958,412.698411,0.259348,0.229980,0.000929,0.003935,0.000772,0.003529,0.002868,0.010157,100
98,277.964485,403.091213,0.146300,0.360551,0.001147,0.004202,0.000948,0.003709,0.003430,0.011131,100


In [10]:
# 计算 RMSE, MAE, MaxAE
RMSE_theta_1 = np.sqrt(np.mean((multi_exp_data['RMSE_theta_1'])**2))
RMSE_theta_2 = np.sqrt(np.mean((multi_exp_data['RMSE_theta_2'])**2))
MAE_theta_1 = np.mean(np.abs(multi_exp_data['MAE_theta_1']))
MAE_theta_2 = np.mean(np.abs(multi_exp_data['MAE_theta_2']))
MaxAE_theta_1 = np.max(np.abs(multi_exp_data['MaxAE_theta_1']))
MaxAE_theta_2 = np.max(np.abs(multi_exp_data['MaxAE_theta_2']))

# save as csv
overall_error = [{
    'RMSE_theta_1': RMSE_theta_1,
    'RMSE_theta_2': RMSE_theta_2,
    'MAE_theta_1': MAE_theta_1,
    'MAE_theta_2': MAE_theta_2,
    'MaxAE_theta_1': MaxAE_theta_1,
    'MaxAE_theta_2': MaxAE_theta_2
}]
overall_error = pd.DataFrame(data=overall_error)
overall_error.to_csv("./data/multi_exps_overall_error.csv", index=False)